# M10.4 · CUDA programlama ve bellek modeli — SIMT, birleştirilmiş erişim, bank çakışması, ayrışma

GPU neden hesaba transistör ayırır, CUDA programının yaşam döngüsü (darboğazı bul → veri paralel kısmı bul → GPU'ya kopyala → hesapla → geri kopyala → CPU ile doğrula → yinele), çekirdek nitelikleri (`__global__`, `__device__`, `__host__`), grid/blok/thread hiyerarşisi, `cudaMalloc`/`cudaMemcpy`, nvcc derleme akışı (PTX, cubin, JIT, hesap yeteneği uyumluluğu), hata denetimi (asenkron hatalar, `cudaGetLastError`), dal ayrışması ve tahminleme (predication), döngü açma. Bellek hiyerarşisi: yazmaç, yerel, paylaşımlı, sabit, doku, global; warp başına bellek işlemi (32 baytlık sektörler) ve birleştirme (coalescing), paylaşımlı bellekte 32 bank ve çakışmalar, doluluk (occupancy). Saf Python'da warp analizörleri, NumPy'da tüm grid için trafik ve çakışma sayımı (naif ve karolu transpoz, dolgu), Triton (CPU yorumlayıcısı) ile kopya/transpoz çekirdekleri, sonda CPU'da önbellek satırı etkisini ölçen gerçek bant genişliği deneyleri.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m10-4-cuda-memory.html](https://m-gonul.github.io/pytorch-lab/m10-4-cuda-memory.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m10_4_cuda_memory.py
CHECKS = {
 "b": "from labcheck import close, no_imports, need, ensure\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"numpy kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"torch\"}))\n\n    def mem():\n        s = need(ns, \"sectors\")\n        ensure([s([4 * k * t for t in range(32)]) for k in (1, 2, 4, 8, 32)] == [4, 8, 16, 32, 32], \"adımlı float okuma\")\n        ensure(s([4 * t + 16 for t in range(32)]) == 5, \"hizasız başlangıç bir sektör fazla\")\n        ensure(s([0] * 32) == 1, \"tüm warp aynı adres\")\n        ensure(s([8 * t for t in range(32)]) == 8, \"float64 ardışık\")\n        b = need(ns, \"bank_degree\")\n        ensure([b([k * t for t in range(32)]) for k in (1, 2, 3, 4, 16, 32, 33)] == [1, 2, 1, 4, 16, 32, 1], \"çakışma dereceleri\")\n        ensure(b([5] * 32) == 1, \"yayın\")\n        ensure(b([t // 2 for t in range(32)]) == 1, \"iki thread aynı kelime: yayın\")\n    R_.test(\"sektör ve bank\", mem)\n\n    def occ():\n        ensure(need(ns, \"divergent_paths\")([t % 2 for t in range(32)]) == 2 and ns[\"divergent_paths\"]([0] * 32) == 1, \"yollar\")\n        o = need(ns, \"occupancy\")\n        rows = []\n        for tpb, regs, smem in ((256, 64, 0), (256, 32, 0), (128, 32, 48 * 1024), (1024, 32, 0), (32, 16, 0), (512, 128, 0)):\n            blocks, occ_, why = o(tpb, regs, smem)\n            rows.append([tpb, regs, smem, blocks, f\"{occ_:.2f}\", why])\n        ensure(o(256, 64, 0)[:2] == (4, 0.5) and o(256, 64, 0)[2] == \"regs\", \"el örneği\")\n        ensure(o(32, 16, 0)[2] == \"blocks\" and o(128, 32, 48 * 1024)[2] == \"smem\", \"sınırlayıcılar\")\n        R_.table(\"Doluluk (SM: 2048 thread, 32 blok, 64 K yazmaç, 100 KB paylaşımlı)\", [\"thread/blok\", \"yazmaç/thread\", \"smem/blok\", \"blok/SM\", \"doluluk\", \"sınırlayan\"], rows,\n                 note=\"Küçük bloklarda blok sınırı, ağır çekirdeklerde yazmaçlar doluluğu düşürür. Yüksek doluluk gecikme gizlemek için bir araçtır, amaç değil (Volkov ödevi).\")\n    R_.test(\"doluluk\", occ)\n",
 "c": "import numpy as np\nfrom labcheck import close, max_loops, need, ensure\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"döngüsüz\", lambda: [max_loops(src, 0, func=f) for f in (\"warp_sectors\", \"warp_bank_degree\", \"transpose_traffic\")])\n\n    def basic():\n        A = np.stack([4 * k * np.arange(32) for k in (1, 2, 4, 8, 32)])\n        close(need(ns, \"warp_sectors\")(A), np.array([4, 8, 16, 32, 32]), what=\"sektörler\")\n        W = np.stack([k * np.arange(32) for k in (1, 2, 3, 4, 16, 32, 33)] + [np.full(32, 7)])\n        close(need(ns, \"warp_bank_degree\")(W), np.array([1, 2, 1, 4, 16, 32, 1, 1]), what=\"çakışmalar\")\n    R_.test(\"warp sayımları\", basic)\n\n    def tr():\n        N = 1024\n        ideal = N * N * 4 // 32\n        rows = []\n        res = {}\n        for name, kind, pad in ((\"kopya\", \"copy\", 0), (\"naif transpoz\", \"naive\", 0), (\"karolu, [32][32]\", \"tiled\", 0), (\"karolu, [32][33]\", \"tiled\", 1)):\n            r, w, b = need(ns, \"transpose_traffic\")(N, kind, 32, pad)\n            res[name] = (r, w, b)\n            rows.append([name, r, w, f\"{(r + w) / (2 * ideal):.2f}\", f\"{b:.0f}\" if kind == \"tiled\" else \"—\"])\n        ensure(res[\"kopya\"][:2] == (ideal, ideal), \"kopya ideal\")\n        ensure(res[\"naif transpoz\"][1] == 8 * ideal, \"naif yazma: her thread ayrı sektör\")\n        ensure(res[\"karolu, [32][32]\"][:2] == (ideal, ideal) and res[\"karolu, [32][32]\"][2] == 32 and res[\"karolu, [32][33]\"][2] == 1, \"karolu: birleştirilmiş, dolgu çakışmayı kaldırır\")\n        R_.table(f\"{N}×{N} float32 transpoz, global trafik (32 B sektör) ve paylaşımlı bellek çakışması\", [\"çekirdek\", \"okuma sektörü\", \"yazma sektörü\", \"trafik / ideal\", \"smem çakışma derecesi\"], rows,\n                 note=\"Naif transpoz 4.5× trafik üretir (yazmalarda 8×). Karolu sürüm trafiği ideale indirir ama dolgusuz karoda sütun okuması 32-yollu çakışmayla 32 kat yavaşlar; tek bir dolgu sütunu bunu çözer.\")\n    R_.test(\"transpoz trafiği\", tr)\n",
 "d": "import torch\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R_):\n    dev = ns.get(\"DEVICE\", \"cpu\")\n    g = torch.Generator().manual_seed(0)\n\n    def cp():\n        for n in (1, 255, 256, 1000):\n            x = torch.randn(n, generator=g).to(dev)\n            close(need(ns, \"copy\")(x).cpu().numpy(), x.cpu().numpy(), what=f\"kopya n = {n}\")\n    R_.test(\"copy\", cp)\n\n    def tr():\n        for M, N in ((16, 16), (37, 50), (64, 96), (1, 7)):\n            x = torch.randn(M, N, generator=g).to(dev)\n            close(need(ns, \"transpose_naive\")(x).cpu().numpy(), x.t().cpu().numpy(), what=f\"naif {M}×{N}\")\n            close(need(ns, \"transpose_tiled\")(x).cpu().numpy(), x.t().cpu().numpy(), what=f\"karolu {M}×{N}\")\n            close(ns[\"transpose_tiled\"](x, 8, 32).cpu().numpy(), x.t().cpu().numpy(), what=f\"karolu 8×32 blok {M}×{N}\")\n        if dev == \"cpu\":\n            R_.note(\"CPU yorumlayıcısında doğruluk sınandı; hız için GPU gerekir. Triton'da tek bir [BM, BN] bloğu yükleyip tl.trans ile yazmak, CUDA'da paylaşımlı bellek karosuyla yaptığımız işi derleyiciye bırakır: Triton yerleşimi (layout) ve gerekirse paylaşımlı bellek dönüşümünü kendisi seçer.\")\n    R_.test(\"transpozlar (kenar blokları)\", tr)\n",
 "e": "import torch\nfrom labcheck import need, ensure, fmt_time\n\n\ndef check(ns, R_):\n    def st():\n        strides = [1, 2, 4, 8, 16, 32, 64]\n        c = need(ns, \"strided_cost\")(1 << 25, strides)\n        ensure(len(c) == len(strides) and c[4] > 2 * c[0], \"adım büyüdükçe eleman başına maliyet artmalı\")\n        R_.table(\"x[::s].sum(), x 32 M float32 (128 MB)\", [\"adım s\", \"eleman başına (ns)\", \"satır başına kullanılan eleman\"], [[s, f\"{v:.2f}\", max(1, 16 // s)] for s, v in zip(strides, c)],\n                 note=\"CPU belleği 64 B satırlarla taşır; s ≥ 16'da her eleman ayrı satır getirir ve maliyet doyar — GPU'da warp başına sektör sayısının karşılığı. (Donanım önceden getirme (prefetch) ardışık adımları kısmen gizler.)\")\n    R_.test(\"adımlı erişim\", st)\n\n    def tp():\n        N = 2048\n        res = need(ns, \"transpose_variants\")(N)\n        ensure(len(res) == 3, \"üç yöntem\")\n        R_.table(f\"{N}×{N} float32 transpoz kopyası (CPU)\", [\"yöntem\", \"süre\"], [[k, fmt_time(v)] for k, v in res.items()],\n                 note=\"PyTorch'un kopya çekirdeği zaten bloklu çalışır; Python döngüsü her satır için çağrı ek yükü ve adımlı yazmayla en yavaşı.\")\n    R_.test(\"transpoz\", tp)\n",
 "x1": "from labcheck import need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        f = need(ns, \"can_run\")\n        cases = [\n            ([(\"sm\", 8, 0)], (8, 6), \"cubin\"), ([(\"sm\", 8, 6)], (8, 0), None), ([(\"sm\", 7, 5)], (8, 0), None),\n            ([(\"compute\", 7, 5)], (8, 9), \"jit\"), ([(\"compute\", 9, 0)], (8, 9), None),\n            ([(\"sm\", 8, 0), (\"compute\", 8, 0)], (9, 0), \"jit\"), ([(\"sm\", 8, 0), (\"compute\", 8, 0)], (8, 6), \"cubin\"),\n        ]\n        rows = []\n        for b, cc, want in cases:\n            got = f(b, cc)\n            ensure(got == want, f\"{b} @ {cc}: {want} beklenirdi, {got}\")\n            rows.append([\", \".join(f\"{k}_{a}{c}\" for k, a, c in b), f\"{cc[0]}.{cc[1]}\", got or \"çalışmaz\"])\n        R_.table(\"Gömülü kod ve cihaz hesap yeteneği\", [\"derleme\", \"cihaz\", \"sonuç\"], rows,\n                 note=\"Yalnız cubin gömülü bir uygulama yeni nesil GPU'da çalışmaz; PTX de gömülürse sürücü JIT ile derler (ilk çalıştırmada gecikme, önbelleğe alınır). Dağıtım için `-gencode` ile birkaç sm_XY + en yüksek compute_XY gömülür.\")\n    R_.test(\"uyumluluk\", t)\n",
 "x2": "import numpy as np\nfrom labcheck import close, max_loops, need, ensure\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"döngüsüz\", lambda: max_loops(src, 0, func=\"warp_cost\"))\n\n    def t():\n        c = np.array([10.0, 30.0])\n        close(need(ns, \"warp_cost\")(np.array([[0] * 32, [1] * 32, [0, 1] * 16]), c), np.array([10, 30, 40.0]), what=\"warp maliyetleri\")\n        th = need(ns, \"kernel_cost\")(1024, \"thread\", c)\n        wp = ns[\"kernel_cost\"](1024, \"warp\", c)\n        ensure(th == 32 * 40 and wp == 16 * 10 + 16 * 30, \"thread vs warp hizalı dallanma\")\n        R_.table(\"1024 thread, iki yol (maliyet 10 ve 30)\", [\"dallanma\", \"toplam warp maliyeti\"], [[\"tid % 2 (her warp ayrışır)\", th], [\"(tid / 32) % 2 (warp hizalı)\", wp], [\"dal yok, yalnız yol 0\", ns[\"kernel_cost\"](1024, \"none\", c)]],\n                 note=\"Aynı iş, aynı sonuç; yalnız thread–veri eşlemesi değişti. İndirgemede (M10.5) bu fikir 'ayrışmayı en aza indiren eşleme'ye dönüşür.\")\n    R_.test(\"ayrışma\", t)\n",
 "x3": "from labcheck import need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        n, G, B = 10000, 4, 64\n        gs = need(ns, \"grid_stride_indices\")(n, G, B)\n        ch = need(ns, \"chunk_indices\")(n, G, B)\n        for a in (gs, ch):\n            flat = sorted(i for lst in a for i in lst)\n            ensure(flat == list(range(n)), \"her eleman tam bir kez\")\n        f1 = need(ns, \"coalesced_fraction\")(gs, 40)\n        f2 = ns[\"coalesced_fraction\"](ch, 40)\n        ensure(f1 == 1.0 and f2 == 0.0, \"grid-stride birleştirilmiş, parça bölme değil\")\n        R_.table(f\"n = {n}, {G} blok × {B} thread\", [\"dağıtım\", \"birleştirilmiş iterasyon oranı\", \"thread başına eleman\"], [[\"grid-stride\", f\"{f1:.2f}\", f\"{len(gs[0])}\"], [\"ardışık parçalar\", f\"{f2:.2f}\", f\"{len(ch[0])}\"]],\n                 note=\"CPU thread'lerine ardışık parça vermek önbellek için iyiydi (M10.3); GPU'da tersi doğru: aynı anda çalışan warp'ın thread'leri komşu adresleri okumalı. Grid-stride döngüsü ayrıca çekirdeği herhangi bir grid boyutuyla doğru kılar.\")\n    R_.test(\"grid-stride\", t)\n"
}

In [ ]:
import importlib, labcheck, checks_m10_4_cuda_memory as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

**GPU ve CPU:** CPU geniş önbellek ve karmaşık kontrolle tek thread gecikmesini düşürür; GPU transistörleri hesap birimlerine ayırır ve gecikmeyi binlerce thread arasında geçiş yaparak gizler (yüksek GFLOP/s ve bellek bant genişliği). **SIMT:** aynı komutu çok thread yürütür; her thread'in kendi yazmaçları vardır.

**CUDA yaklaşımı:** (1) darboğazı bul, (2) veri paralel kısmı bul (vektör/matris işlemleri, konvolüsyon, görüntü/video işleme…), (3) girdiyi GPU'ya kopyala, (4) GPU'da hesapla, (5) sonucu geri kopyala, (6) CPU sonucuyla doğrula, (7) yinele. Bellek: `cudaMalloc(&p, bayt)`, `cudaMemcpy(dst, src, bayt, cudaMemcpyHostToDevice…)` (engelleyici), `cudaFree`. Fonksiyon nitelikleri: `__global__` (cihazda çalışır, konaktan çağrılır), `__device__` (cihazdan), `__host__`. Başlatma `kernel<<<grid, block>>>(…)`; for döngüsü yok: her thread `blockIdx`, `blockDim`, `threadIdx` ile kendi verisini seçer.

**Derleme:** nvcc cihaz kodunu **PTX** (sanal ISA) ve/veya **cubin**'e (belirli mimari) derler. cubin `sm_XY`, aynı ana sürümde X.z ≥ X.y cihazlarda çalışır; PTX `compute_XY` sürücünün **JIT** derleyicisiyle ≥ X.Y her cihazda çalışır (ileri uyum). `-arch`/`-gencode` hem cubin hem PTX gömebilir.

**Hatalar:** tüm çalışma zamanı fonksiyonları hata kodu döndürür; çekirdek başlatma döndürmez. `cudaPeekAtLastError()` (sıfırlamaz) / `cudaGetLastError()` (sıfırlar) başlatma hatalarını, `cudaDeviceSynchronize()` sonrası kontrol asenkron (çalışma sırasındaki) hataları yakalar.

**Ayrışma:** bir warp'ın thread'leri farklı dallara giderse yollar sırayla yürür (aktif olmayan thread'ler maskelenir). Kısa dallarda derleyici **tahminleme** (predication) kullanır: her iki yolun komutları yürür, yalnız koşulu doğru olanlar yazar. Koşulu warp sınırlarına hizala (`threadIdx.x / 32`). **Döngü açma:** `#pragma unroll [n]`; işaretli döngü sayaçları derleyiciye taşma varsayımı yapmama özgürlüğü verir (işaretsiz sarmalama tanımlıdır).

**Bellek hiyerarşisi:** yazmaçlar (thread), yerel (thread'e özel ama DRAM'de: taşan yazmaçlar), **paylaşımlı** (blok, çip üstünde, ~100 KB/SM, 32 bank × 4 B), **sabit** (salt okunur, tüm warp aynı adresi okursa yayın), doku (2B önbellek), **global** (DRAM). Global erişim warp başına **32 B sektörlerle** yapılır: ardışık 32 float = 4 sektör (birleştirilmiş); adım büyüdükçe sektör sayısı 32'ye çıkar. Paylaşımlı bellekte aynı banktaki farklı adresler sıralanır (aynı adres yayın). **Doluluk:** SM'de aynı anda çalışan warp'lar / en çok warp; thread, yazmaç, paylaşımlı bellek ve blok sınırlarından en darı belirler.

Bu ortamda GPU yok: analizleri sayarak, çekirdekleri Triton'un CPU yorumlayıcısında çalıştırarak ve önbellek satırının CPU'daki karşılığını ölçerek öğreniyoruz.

## 1 · Elle

Adres $4st$: $s=1$ → 0…124 B → 4 sektör; $s=2$ → 0…248 → 8; $s=32$ → her biri 128 B ara → 32 sektör (aktarılan 1024 B'nin yalnız 128'i kullanılır).

Bank $=$ kelime adresi mod 32: $2t$ → 0, 2, …, 30, 0, … (her banka 2 adres); $32t$ → hep bank 0; $33t$ → $t$ mod 32, hepsi farklı.

Doluluk: yazmaç sınırı $65536/(64\cdot256)=4$ blok, thread sınırı 8 → 4 blok × 256 = 1024 / 2048.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**Bir warp (32 thread) float32 okuyor, taban adres 128 B hizalı; kaç 32 B sektör?**

- `s1`: thread t → eleman t
- `s2`: thread t → eleman 2t
- `s32`: thread t → eleman 32t (satır-öncelikli 32 sütunlu matrisin bir sütunu)

**Paylaşımlı bellek, 32 bank × 4 B: warp'ın çakışma derecesi**

- `b2`: s[2t]
- `b32`: s[32t] (32×32 karonun bir sütunu)
- `b33`: s[33t] (32×33 dolgulu karo)

**Doluluk ve ayrışma**

- `occ`: SM: 2048 thread, 65 536 yazmaç; blok 256 thread, thread başına 64 yazmaç → doluluk
- `div`: `if (threadIdx.x % 2)` ile her warp kaç yolu sırayla yürütür?

In [ ]:
cevaplar = dict(
    s1=None,
    s2=None,
    s32=None,
    b2=None,
    b32=None,
    b33=None,
    occ=None,
    div=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"s1":[4,0],"s2":[8,0],"s32":[32,0],"b2":[2,0],"b32":[32,0],"b33":[1,0],"occ":[0.5,0],"div":[2,0]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python warp analizörleri.

- `sectors(byte_addrs, sector=32)` — bir warp'ın dokunduğu farklı sektör sayısı.
- `bank_degree(word_addrs, n_banks=32)` — çakışma derecesi: bank başına **farklı** adres sayısının en büyüğü (aynı adres yayın: 1 sayılır).
- `divergent_paths(conds)` — warp'taki thread'lerin dal koşullarından kaç farklı yol yürütüleceği.
- `occupancy(tpb, regs, smem, max_threads=2048, max_blocks=32, regs_sm=65536, smem_sm=102400)` → `(SM başına blok, doluluk oranı, sınırlayan etken)`; etken `"threads"`, `"regs"`, `"smem"` ya da `"blocks"`.

In [ ]:
%%writefile stage_b.py
def sectors(byte_addrs, sector=32):
    ...


def bank_degree(word_addrs, n_banks=32):
    ...


def divergent_paths(conds):
    ...


def occupancy(tpb, regs, smem, max_threads=2048, max_blocks=32, regs_sm=65536, smem_sm=102400):
    ...


print(sectors([4 * t for t in range(32)]), bank_degree([33 * t for t in range(32)]), occupancy(256, 64, 0))

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

NumPy, döngüsüz: tüm grid'in warp'larını bir dizide sayar.

- `warp_sectors(addr)` — `addr (n_warps, 32)` bayt adresleri → warp başına sektör sayısı (satır içi sıralama + değişim sayımı).
- `warp_bank_degree(words)` — `(n_warps, 32)` kelime adresleri → çakışma derecesi (benzersiz adresler, `np.add.at` ile bank sayımı).
- `transpose_traffic(N, kind, T=32, pad=0)` → `(okuma sektörleri, yazma sektörleri, paylaşımlı okuma çakışma derecesi ortalaması)`; `kind`: `"copy"`, `"naive"`, `"tiled"`. Thread (y, x) blok içinde x-ana sırada warp'lara bölünür; `"tiled"`: blok T×T, karo `tile[ty][tx]`'e birleştirilmiş okunur, `tile[tx][ty]`'den (kelime adresi `tx·(T + pad) + ty`) okunup birleştirilmiş yazılır. N, T'nin katı, float32.

In [ ]:
%%writefile stage_c.py
import numpy as np


def warp_sectors(addr):
    ...


def warp_bank_degree(words):
    ...


def transpose_traffic(N, kind, T=32, pad=0):
    ...


print(transpose_traffic(256, "naive"), transpose_traffic(256, "tiled", pad=1))

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

Triton (GPU yoksa yorumlayıcı).

- `copy(x)` — 1B blok kopyası (maske ile).
- `transpose_naive(x)` — her program bir satır parçası okur (bitişik), devrik konuma (adımlı) yazar.
- `transpose_tiled(x, BM=16, BN=16)` — 2B program ızgarası; `[BM, BN]` blok yükle, `tl.trans`, devrik bloğa yaz; kenarlarda maske.

Not: Triton paylaşımlı belleği ve birleştirmeyi kendisi yönetir; burada çekirdeğin hangi adresleri ürettiğini c'deki modelle düşünüyoruz.

In [ ]:
%%writefile stage_d.py
import os
import torch

if not torch.cuda.is_available():
    os.environ.setdefault("TRITON_INTERPRET", "1")
import triton
import triton.language as tl

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


@triton.jit
def copy_kernel(X, Y, n, BLOCK: tl.constexpr):
    pass  # TODO


def copy(x):
    ...


def transpose_naive(x):
    ...


def transpose_tiled(x, BM=16, BN=16):
    ...


print(copy(torch.arange(10.0)))

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

**CPU'da aynı fizik:** önbellek satırı (64 B) GPU sektörünün karşılığı.

- `strided_cost(n, strides, repeat=3)` → her adım için **eleman başına** süre (ns): `x[::s].sum()` (float32, x büyük ve önbelleğe sığmaz).
- `transpose_variants(N)` → sözlük: `x.t().contiguous()`, karo karo (T = 32) `reshape`/`permute` ile devrik kopya, satır satır Python döngüsüyle (`out[:, i] = x[i]`) — süreler.

In [ ]:
%%writefile stage_e.py
import time
import torch


def strided_cost(n, strides, repeat=3):
    ...


def transpose_variants(N):
    ...

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Ek alıştırmalar

Beş aşamanın dışındaki pratikler; her biri kendi kontrolüyle.

### E1 · nvcc, PTX ve hesap yeteneği uyumluluğu

- `can_run(build, cc)` — `build` gömülü kod listesi: `("sm", 8, 0)` cubin ya da `("compute", 7, 5)` PTX; `cc = (ana, alt)` cihaz. cubin: aynı ana sürüm ve alt ≥; PTX: cihaz ≥ PTX sürümü (JIT). Çalışırsa hangi yolla: `"cubin"`, `"jit"` ya da `None`.

In [ ]:
%%writefile extra_x1.py
def can_run(build, cc):
    ...


print(can_run([("sm", 8, 0), ("compute", 8, 0)], (8, 6)))

In [ ]:
# Kontrol (E1)
labcheck.run_file("extra_x1.py", CHECKS["x1"])

### E2 · Dal ayrışmasının maliyet modeli

Bir warp'ın maliyet modeli: her yol maliyeti, warp'ta o yolu seçen en az bir thread varsa ödenir (yollar sıralanır).

- `warp_cost(path_ids, costs)` — `path_ids (n_warps, 32)`, `costs` yol → maliyet dizisi; warp başına toplam (NumPy, döngüsüz: one-hot `any`).
- `kernel_cost(n_threads, branch, costs)` — `branch` = `"thread"` (`tid % 2`), `"warp"` (`(tid // 32) % 2`), `"none"`.

In [ ]:
%%writefile extra_x2.py
import numpy as np


def warp_cost(path_ids, costs):
    ...


def kernel_cost(n_threads, branch, costs):
    ...


print(kernel_cost(1024, "thread", np.array([10, 30])))

In [ ]:
# Kontrol (E2)
labcheck.run_file("extra_x2.py", CHECKS["x2"])

### E3 · Grid-stride döngüsü

- `grid_stride_indices(n, grid, block)` → her thread'in işlediği indeks listesi (`for i = tid; i < n; i += grid·block`).
- `chunk_indices(n, grid, block)` → her thread ardışık bir parça.
- `coalesced_fraction(assign, n_iter)` — her iterasyonda warp'ın erişimleri ardışık mı (oran).

In [ ]:
%%writefile extra_x3.py
def grid_stride_indices(n, grid, block):
    ...


def chunk_indices(n, grid, block):
    ...


def coalesced_fraction(assign, n_iter):
    ...


print(grid_stride_indices(10, 1, 4))

In [ ]:
# Kontrol (E3)
labcheck.run_file("extra_x3.py", CHECKS["x3"])

## Makale ödevi: Karolu transpoz ve bant genişliği (Harris, NVIDIA blog)

[Harris — An Efficient Matrix Transpose in CUDA C/C++](https://developer.nvidia.com/blog/efficient-matrix-transpose-cuda-cc/)

GPU'lu bir ortamda (Colab) CUDA C++ ile kopya, naif transpoz, paylaşımlı bellekli karolu transpoz ve dolgulu karolu transpozu yaz; `cudaEvent` ile zamanla, efektif bant genişliğini (GB/s) hesapla ve cihazın tepe değeriyle karşılaştır. Nsight Compute ile her çekirdeğin `l1tex__t_sectors_pipe_lsu_mem_global_op_ld/st` ve `l1tex__data_bank_conflicts_pipe_lsu_mem_shared` sayaçlarını oku ve c aşamasındaki modelin öngörüleriyle karşılaştır.


**Adımlar**

1. Hata denetimi makrosu (`CUDA_CHECK`) ve çekirdek sonrası `cudaGetLastError` + `cudaDeviceSynchronize`.
2. TILE_DIM 32, BLOCK_ROWS 8 (thread başına 4 eleman).

**Kabul ölçütleri**

- Dolgulu karolu transpoz kopyanın ≥ %80'i.
- Profil sayaçları c aşamasının sektör ve çakışma sayılarıyla ±%5 uyumlu.

## Makale ödevi: Volkov (2010): Daha düşük dolulukla daha iyi performans

[Volkov — Better Performance at Lower Occupancy (GTC 2010)](https://www.nvidia.com/content/GTC-2010/pdfs/2238_GTC2010.pdf)

Sunumdaki deneyleri yeniden üret: thread başına bağımsız işlem sayısını (ILP) artırıp dolulukla birlikte azaltarak a·x + b ve bellek kopyası çekirdeklerinde verimi ölç. b aşamasındaki doluluk hesaplayıcısıyla her yapılandırmanın doluluğunu raporla; Little yasasıyla (gecikme × verim = eşzamanlılık) sonucu açıkla.


**Adımlar**

1. Thread başına 1, 2, 4, 8 bağımsız yükleme; blok başına 64–1024 thread.

**Kabul ölçütleri**

- %25 dolulukla tepe verime yakın bir yapılandırma bulunur.

## Kaynaklar

- [NVIDIA — CUDA C++ Programming Guide](https://docs.nvidia.com/cuda/cuda-c-programming-guide/) — NVIDIA
- [NVIDIA — CUDA C++ Best Practices Guide (bellek erişimi, doluluk)](https://docs.nvidia.com/cuda/cuda-c-best-practices-guide/) — NVIDIA
- [Hwu, Kirk, El Hajj — Programming Massively Parallel Processors (4. baskı), Bölüm 2–6](https://www.elsevier.com/books/programming-massively-parallel-processors/hwu/978-0-323-91231-0) — ©
- [Harris — An Efficient Matrix Transpose in CUDA C/C++ (NVIDIA Developer Blog)](https://developer.nvidia.com/blog/efficient-matrix-transpose-cuda-cc/) — ©
- [Volkov — Better Performance at Lower Occupancy (GTC 2010)](https://www.nvidia.com/content/GTC-2010/pdfs/2238_GTC2010.pdf) — ©
- [Triton — Tutorials](https://triton-lang.org/main/getting-started/tutorials/index.html) — MIT